In [2]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.6 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=659972daab224e877414f76bc570bb6932200bde0ef7d79861d106e4577b7493
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [ ]:
import uuid
import pandas as pd
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
import re
from langdetect import detect
from nltk.tokenize import RegexpTokenizer
from gensim.parsing.preprocessing import strip_punctuation
import os
import pyarrow as pa
import pyarrow.parquet as pq
import ast  

In [ ]:
# with open("data/missing_ids.txt", "r") as f:
#     missing = f.read()
#     missing = ast.literal_eval(missing) 
    
# with open("data/relevant_ids.txt", "r") as f:
#     relevant = f.read()
#     relevant = ast.literal_eval(relevant) 


# # Ensure both are sets
# missing = set(missing)
# relevant = set(relevant)

# # Compute the difference
# ids = relevant - missing

# with open("data/valid_ids.txt", "w") as f:
#     f.write(str(ids))


In [ ]:
def _unify_text(text):
    """Unify text e.g. remove URLs, lowercase, etc.
    """
    
    if text != text:
        return text

    
    #text = unicodedata.normalize('NFKD', text) # "déjà vu" → "deja vu"

    #text = re.sub(r"#[A-Za-z0-9_]+", "", text)  # remove #hashtag
    text = re.sub(r"RT\ ", " ", text)  # remove 'RT' from tweets
    text = re.sub(r"@[A-Za-z0-9_]+", "@user", text)  # remove @user

    text = re.sub(r'\*\*',"",text) # "remove bold asterisks"
    text =  re.sub(r'_{2,}', '_', text) # remove multiple "_" 
    
    text = re.sub(r'\[.*?\]\(https?://[^\)]+\)', '', text) # remove attachement links
    text = re.sub(r'https?://\S+', '', text) # remove remaining links
    text = re.sub(r"https?://[A-Za-z0-9./]+", " ", text)  # remove links
    
    text = re.sub("\t", " ", text)  # remove tab
    text = re.sub("\n", " ", text)  # remove newlines
    text = re.sub("\r", " ", text)  # remove \r type newlines
    text = re.sub(r" +", " ", text)  # remove multiple whitespaces
    text = re.sub(r"linebreak", "", text)  # remove linebreaks
    text = text.strip() # Remove leading whitespace
    return text

def split_into_sentences(text):
    """
    Splits an text into sentences.

    Args:
        text (str): The text to be split.

    Returns:
        list: The list of sentences.
    """
    sentences = sent_tokenize(text)

    def en_sentence(sentence):
        if detect(sentence) == "en":
            return True
        else:
            return False
            

    sentences = [
        sentence for sentence in sentences if en_sentence(sentence)
    ]
    return sentences

# Split message into sentences

In [ ]:
# Load dataframe
df = pd.read_parquet("/kaggle/working/messages_en_split4.parquet")

# Setup Parquet writer
output_path = "sentences_en.parquet"
writer = None

# Parameters
batch = []
batch_size = 10000  # Adjust depending on available memory


In [ ]:
for _, row in tqdm(df.iterrows(), total=len(df)):
    message = row["message"]
    ch_id = row["id"]
    sentences = split_into_sentences(message)
    sentence_ids = [str(uuid.uuid4()) for _ in range(len(sentences))]

    batch.extend(
        {"sentence": s, "ch_id": ch_id, "sentence_id": sid}
        for s, sid in zip(sentences, sentence_ids)
    )

    # Write batch to disk
    if len(batch) >= batch_size:
        batch_df = pd.DataFrame(batch)
        table = pa.Table.from_pandas(batch_df)
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema)
        writer.write_table(table)
        batch = []

# Write remaining
if batch:
    batch_df = pd.DataFrame(batch)
    table = pa.Table.from_pandas(batch_df)
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema)
    writer.write_table(table)

if writer:
    writer.close()

# Pre-Process Sentences

In [ ]:
df = pd.read("")

In [4]:
df["sentence"] = df["sentence"].apply(_unify_text)
df = df[df.sentence != ""]
df = df[~df.sentence.isna()]
df = df[~df.sentence.duplicated()]

,sentence,ch_id,sentence_id,proc_sentence
0,Cop or drop ✅💯#AF1,1195156359,ea5a15e9-f029-45cd-a63c-1a14f9f33952,Cop or drop ✅💯#AF1
1,Off-white jeans available,1195156359,3fca1dfc-4f3a-4e60-afd9-09aac92007e8,Off-white jeans available
2,Nike airforce 1 white \nEu :41-45,1195156359,f778e477-694d-4187-9a75-ede36f945e2c,Nike airforce 1 white Eu :41-45
3,Daily paper pants,1195156359,77278ff8-ecb8-4701-a779-aa8192bf0eeb,Daily paper pants
4,EU size 41-42,1195156359,fe5f2822-2b94-49d8-b2e9-bdac80f6f254,EU size 41-42
...,...,...,...,...
12321545,VODACOM BLOCKED!!,1653932750,5c2526cc-3532-4bb7-81ea-473222e34b0a,VODACOM BLOCKED!!
12321546,CELL C BLOCKED!!,1653932750,adf5efde-3306-4455-9c83-ada261c3be01,CELL C BLOCKED!!
12321547,MTN DROPED HERE\n\n@WE_PLUG_THE_UNPLUGED\n\n\n...,1653932750,453d2ead-da26-42ad-a503-428ba2f859b1,MTN DROPED HERE @WE_PLUG_THE_UNPLUGED JOIN HER...
12321548,IF WANT TO FUCK YOUR SELF JOIN HERE\n\n\n@Fuck...,1653932750,cf7c49db-61a3-40b3-ab50-f7b5835ebfdc,IF WANT TO FUCK YOUR SELF JOIN HERE @Fuck_your...


In [ ]:
df.to_parquet("sentences_cleaned.parquet")